In [23]:
import sys
sys.path.append("..")

from pathlib import Path
from src.ingestion.document_inspector import inspect_document
from src.parsing.document_parser import parse_document
from src.cleaning.document_cleaner import clean_document
from src.structure.line_grouper import group_spans_into_lines
from src.structure.chapter_detector import (detect_explicit_chapter_candidates, 
                                            detect_chapter_candidates, 
                                            compose_chapter_heading, 
                                            collect_document_lines, 
                                            build_chapter,
                                            compose_chapter_headings,
                                            get_chapter_number,
                                            group_chapter_sequences,
                                            score_chapter_sequence,
                                            build_candidate_context)
from src.chunking.text_chunker import TextChunker
from src.embedding.text_embedding import TextEmbedding

In [24]:
RAW_DATA_DIR = Path("../data/raw")

pdf_files = sorted(RAW_DATA_DIR.glob("*.pdf"))

for pdf_file in pdf_files[:5]:
    print(pdf_file.name)

Atomic habits ( PDFDrive ).pdf
attention is all you need.pdf
bert research paper copy.pdf
bert research paper.pdf
deep-learning-material-dept-ece-ase-blr-1.pdf


In [25]:
# Inspecting the first document
path = Path("../data/raw/Atomic habits ( PDFDrive ).pdf")
inspect = inspect_document(path)
inspect

{'filename': 'Atomic habits ( PDFDrive ).pdf',
 'filepath': '..\\data\\raw\\Atomic habits ( PDFDrive ).pdf',
 'page_count': 256,
 'metadata': {'format': 'PDF 1.4',
  'title': 'Atomic habits \\( PDFDrive.com \\).pdf',
  'author': 'James Clear',
  'subject': '',
  'keywords': '',
  'creator': 'calibre 3.48.0 [https://calibre-ebook.com]',
  'producer': 'calibre 3.48.0 [https://calibre-ebook.com]',
  'creationDate': "D:20200430184622+00'00'",
  'modDate': '',
  'trapped': '',
  'encryption': None},
 'pages': [{'page_number': 1, 'text': '', 'text_length': 0},
  {'page_number': 2, 'text': '', 'text_length': 0},
  {'page_number': 3,
   'text': 'AN IMPRINT OF PENGUIN RANDOM HOUSE LLC\n375 Hudson Street\nNew York, New York 10014\nCopyright © 2018 by James Clear\nPenguin supports copyright. Copyright fuels creativity, encourages diverse voices, promotes free speech, and creates a vibrant culture. Thank you for buying an authorized edition of this book and for\ncomplying with copyright laws by no

In [26]:
# Document parser

file_path = Path("../data/raw/Atomic habits ( PDFDrive ).pdf")
parsed_document = parse_document(file_path)
print(parsed_document)

Document(filename='Atomic habits ( PDFDrive ).pdf', filepath=WindowsPath('../data/raw/Atomic habits ( PDFDrive ).pdf'), page_count=256, metadata={'format': 'PDF 1.4', 'title': 'Atomic habits \\( PDFDrive.com \\).pdf', 'author': 'James Clear', 'subject': '', 'keywords': '', 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]', 'producer': 'calibre 3.48.0 [https://calibre-ebook.com]', 'creationDate': "D:20200430184622+00'00'", 'modDate': '', 'trapped': '', 'encryption': None}, pages=[Page(page_number=1, text='', spans=[]), Page(page_number=2, text='', spans=[]), Page(page_number=3, text='AN IMPRINT OF PENGUIN RANDOM HOUSE LLC\n375 Hudson Street\nNew York, New York 10014\nCopyright © 2018 by James Clear\nPenguin supports copyright. Copyright fuels creativity, encourages diverse voices, promotes free speech, and creates a vibrant culture. Thank you for buying an authorized edition of this book and for\ncomplying with copyright laws by not reproducing, scanning, or distributing any part 

In [27]:
# checking page spans
page = parsed_document.pages[30]

print(page.spans[0])

TextSpan(text='FIGURE 4: With outcome-based habits, the focus is on what you want to achieve. With identity-based habits, the focus is on who you wish to become.', page_number=31, font='LiberationSerif', font_size=5.760000228881836, flags=4, bbox=(100.800048828125, 476.5687255859375, 448.78509521484375, 482.3287353515625))


In [28]:
# Cleaning and normalizing parsed document

cleaned_document = clean_document(parsed_document)

print(cleaned_document.pages[15])

Page(page_number=16, text='1\nThe Surprising Power of Atomic Habits\nTHE FATE OF British Cycling changed one day in 2003. The organization, which was\nthe governing body for professional cycling in Great Britain, had recently hired\nDave Brailsford as its new performance director. At the time, professional\ncyclists in Great Britain had endured nearly one hundred years of mediocrity.\nSince 1908, British riders had won just a single gold medal at the Olympic\nGames, and they had fared even worse in cycling’s biggest race, the Tour de\nFrance. In 110 years, no British cyclist had ever won the event.\nIn fact, the performance of British riders had been so underwhelming that one\nof the top bike manufacturers in Europe refused to sell bikes to the team because\nthey were afraid that it would hurt sales if other professionals saw the Brits using\ntheir gear.\nBrailsford had been hired to put British Cycling on a new trajectory. What\nmade him different from previous coaches was his relentl

In [29]:
lines = group_spans_into_lines(cleaned_document.pages[10])

for line in lines:
    print(repr(line.text))

'injury, when I began college at Denison University. It was a new beginning, and'
'it was the place where I would discover the surprising power of small habits for'
'the first time.'
'HOW I LEARNED ABOUT HABITS'
'Attending Denison was one of the best decisions of my life. I earned a spot on'
'the baseball team and, although I was at the bottom of the roster as a freshman, I'
'was thrilled. Despite the chaos of my high school years, I had managed to'
'become a college athlete.'
'I wasn’t going to be starting on the baseball team anytime soon, so I focused'
'on getting my life in order. While my peers stayed up late and played video'
'games, I built good sleep habits and went to bed early each night. In the messy'
'world of a college dorm, I made a point to keep my room neat and tidy. These'
'improvements were minor, but they gave me a sense of control over my life. I'
'started to feel confident again. And this growing belief in myself rippled into the'
'classroom as I improved my study 

In [30]:
detect_chapter = detect_explicit_chapter_candidates(lines)

print(detect_chapter)

[]


In [31]:
chapter_candidates = detect_chapter_candidates(parsed_document)
chapter_candidate = chapter_candidates[10]

In [32]:
composed_chapter_candidate = compose_chapter_heading(chapter_candidate, lines)
composed_chapter_candidate

ChapterCandidate(lines=(TextLine(text='11', page_number=113, spans=(TextSpan(text='11', page_number=113, font='LiberationSerif', font_size=23.760000228881836, flags=4, bbox=(294.1199951171875, 164.0459442138672, 317.8800048828125, 187.80593872070312)),)),), page_number=113, score=0.8, reasons=('standalone numeric heading', 'visually prominent typography'))

In [33]:
all_document_lines = collect_document_lines(cleaned_document)
all_document_lines

[TextLine(text='AN IMPRINT OF PENGUIN RANDOM HOUSE LLC', page_number=3, spans=(TextSpan(text='AN IMPRINT OF PENGUIN RANDOM HOUSE LLC', page_number=3, font='LiberationSerif', font_size=5.760000228881836, flags=4, bbox=(241.63877868652344, 113.68871307373047, 370.366943359375, 119.44871520996094)),)),
 TextLine(text='375 Hudson Street', page_number=3, spans=(TextSpan(text='375 Hudson Street', page_number=3, font='LiberationSerif', font_size=5.760000228881836, flags=4, bbox=(284.58001708984375, 120.88871765136719, 327.4237976074219, 126.64871978759766)),)),
 TextLine(text='New York, New York 10014', page_number=3, spans=(TextSpan(text='New York, New York 10014', page_number=3, font='LiberationSerif', font_size=5.760000228881836, flags=4, bbox=(272.52001953125, 128.08872985839844, 339.4800109863281, 133.84872436523438)),)),
 TextLine(text='Copyright © 2018 by James Clear', page_number=3, spans=(TextSpan(text='Copyright © 2018 by James Clear', page_number=3, font='LiberationSerif', font_siz

In [34]:
composed_candidates = compose_chapter_headings(candidates=chapter_candidates, lines=lines)
composed_candidates

[ChapterCandidate(lines=(TextLine(text='1', page_number=16, spans=(TextSpan(text='1', page_number=16, font='LiberationSerif', font_size=23.760000228881836, flags=4, bbox=(300.05999755859375, 164.0459442138672, 311.94000244140625, 187.80593872070312)),)),), page_number=16, score=0.8, reasons=('standalone numeric heading', 'visually prominent typography')),
 ChapterCandidate(lines=(TextLine(text='2', page_number=28, spans=(TextSpan(text='2', page_number=28, font='LiberationSerif', font_size=23.760000228881836, flags=4, bbox=(300.05999755859375, 164.0459442138672, 311.94000244140625, 187.80593872070312)),)),), page_number=28, score=0.8, reasons=('standalone numeric heading', 'visually prominent typography')),
 ChapterCandidate(lines=(TextLine(text='3', page_number=40, spans=(TextSpan(text='3', page_number=40, font='LiberationSerif', font_size=23.760000228881836, flags=4, bbox=(300.05999755859375, 164.0459442138672, 311.94000244140625, 187.80593872070312)),)),), page_number=40, score=0.8, 

In [35]:
chapter_sequences = group_chapter_sequences(composed_candidates)

print("Number of sequences:", len(chapter_sequences))

for i, sequence in enumerate(chapter_sequences, start=1):
    print(f"\nSequence {i}")
    print(f"Length: {len(sequence)}")

    for candidate in sequence:
        number = get_chapter_number(candidate)

        print(
            f"Page: {candidate.page_number} | "
            f"Number: {number} | "
            f"Text: {[line.text for line in candidate.lines]}"
        )

Number of sequences: 2

Sequence 1
Length: 20
Page: 16 | Number: 1 | Text: ['1']
Page: 28 | Number: 2 | Text: ['2']
Page: 40 | Number: 3 | Text: ['3']
Page: 52 | Number: 4 | Text: ['4']
Page: 59 | Number: 5 | Text: ['5']
Page: 69 | Number: 6 | Text: ['6']
Page: 78 | Number: 7 | Text: ['7']
Page: 84 | Number: 8 | Text: ['8']
Page: 94 | Number: 9 | Text: ['9']
Page: 103 | Number: 10 | Text: ['10']
Page: 113 | Number: 11 | Text: ['11']
Page: 119 | Number: 12 | Text: ['12']
Page: 127 | Number: 13 | Text: ['13']
Page: 135 | Number: 14 | Text: ['14']
Page: 144 | Number: 15 | Text: ['15']
Page: 152 | Number: 16 | Text: ['16']
Page: 160 | Number: 17 | Text: ['17']
Page: 168 | Number: 18 | Text: ['18']
Page: 177 | Number: 19 | Text: ['19']
Page: 184 | Number: 20 | Text: ['20']

Sequence 2
Length: 20
Page: 209 | Number: 1 | Text: ['CHAPTER 1']
Page: 210 | Number: 2 | Text: ['CHAPTER 2']
Page: 211 | Number: 3 | Text: ['CHAPTER 3']
Page: 212 | Number: 4 | Text: ['CHAPTER 4']
Page: 213 | Number: 5 

In [36]:
for i, sequence in enumerate(chapter_sequences, start=1):
    score = score_chapter_sequence(sequence)

    print(f"Sequence {i}")
    print(f"Score: {score:.3f}")
    print(
        "Numbers:",
        [get_chapter_number(candidate) for candidate in sequence],
    )
    print(
        "Pages:",
        [candidate.page_number for candidate in sequence],
    )
    print()

Sequence 1
Score: 1.000
Numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Pages: [16, 28, 40, 52, 59, 69, 78, 84, 94, 103, 113, 119, 127, 135, 144, 152, 160, 168, 177, 184]

Sequence 2
Score: 1.000
Numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Pages: [209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228]



In [37]:
chapter = build_chapter(lines=all_document_lines, candidates=composed_candidates)
chapter

[Chapter(heading=ChapterCandidate(lines=(TextLine(text='1', page_number=16, spans=(TextSpan(text='1', page_number=16, font='LiberationSerif', font_size=23.760000228881836, flags=4, bbox=(300.05999755859375, 164.0459442138672, 311.94000244140625, 187.80593872070312)),)),), page_number=16, score=0.8, reasons=('standalone numeric heading', 'visually prominent typography')), content=(TextLine(text='The Surprising Power of Atomic Habits', page_number=16, spans=(TextSpan(text='The Surprising Power of Atomic Habits', page_number=16, font='LiberationSerif', font_size=23.760000228881836, flags=4, bbox=(116.33625030517578, 206.5259552001953, 495.6589050292969, 230.28594970703125)),)), TextLine(text='T', page_number=16, spans=(TextSpan(text='T', page_number=16, font='LiberationSerif-Bold', font_size=26.64000129699707, flags=20, bbox=(72.0, 249.8037109375, 89.76888275146484, 276.4437255859375)),)), TextLine(text='British Cycling changed one day in 2003. The organization, which was', page_number=16

In [38]:
candidate = composed_candidates[19]

context = build_candidate_context(
    candidate=candidate,
    lines=all_document_lines,
)

In [39]:
print("Candidate:")
for line in context.candidate.lines:
    print(" ", repr(line.text))

print("\nPrevious:")
print(
    repr(context.previous_line.text)
    if context.previous_line
    else None
)

print("\nNext:")
print(
    repr(context.next_line.text)
    if context.next_line
    else None
)

print("\nFollowing:")
for line in context.following_lines:
    print(" ", repr(line.text))

Candidate:
  '20'

Previous:
'Professionals stick to the schedule; amateurs let life get in the way.'

Next:
'The Downside of Creating Good Habits'

Following:
  'The Downside of Creating Good Habits'
  'H'
  'In chess, it is only after the basic movements of the'


In [40]:
# Text chunker

page = cleaned_document.pages[15]


chunker = TextChunker()

for chunk in chunker.chunk_page(page):
    print(chunk)

TextChunk(chunk_id='page_16chunk_0', text='1 The Surprising Power of Atomic Habits THE FATE OF British Cycling changed one day in 2003. The organization, which was the governing body for professional cycling in Great Britain, had recently hired Dave Brailsford as its new performance director. At the time, professional cyclists in Great Britain had endured nearly one hundred years of mediocrity. Since 1908, British riders had won just a single gold medal at the Olympic Games, and they had fared even worse in cycling’s biggest race, the Tour de France. In 110 years, no British cyclist had ever won the event. In fact, the performance of British riders had been so underwhelming that one of the top bike manufacturers in Europe refused to sell bikes to the team because they were afraid that it would hurt sales if other professionals saw the Brits using their gear. Brailsford had been hired to put British Cycling on a new trajectory. What made him different from previous coaches was his relen

In [41]:

chunked_document = chunker.chunk(cleaned_document)


for chunk in chunked_document:
    print(chunk)

TextChunk(chunk_id='page_3chunk_0', text='AN IMPRINT OF PENGUIN RANDOM HOUSE LLC 375 Hudson Street New York, New York 10014 Copyright © 2018 by James Clear Penguin supports copyright. Copyright fuels creativity, encourages diverse voices, promotes free speech, and creates a vibrant culture. Thank you for buying an authorized edition of this book and for complying with copyright laws by not reproducing, scanning, or distributing any part of it in any form without permission. You are supporting writers and allowing Penguin to continue to publish books for every reader.', page_number=3, chunk_index=0, word_count=84)
TextChunk(chunk_id='page_4chunk_0', text='Ebook ISBN 9780735211308 While the author has made every effort to provide accurate Internet addresses at the time of publication, neither the publisher nor the author assumes any responsibility for errors, or for changes that occur after publication. Further, the publisher does not have any control over and does not assume any respons

In [42]:
embed = TextEmbedding()

embed.load_model()

Load embedding model BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully
Load embedding model BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [21]:
embedded_chunks = embed.embed(chunked_document)

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

In [44]:
print(embedded_chunks[0].chunk.text)

AN IMPRINT OF PENGUIN RANDOM HOUSE LLC 375 Hudson Street New York, New York 10014 Copyright © 2018 by James Clear Penguin supports copyright. Copyright fuels creativity, encourages diverse voices, promotes free speech, and creates a vibrant culture. Thank you for buying an authorized edition of this book and for complying with copyright laws by not reproducing, scanning, or distributing any part of it in any form without permission. You are supporting writers and allowing Penguin to continue to publish books for every reader.
